# Kouman IA — Notebook 2 : Entraînement CPT Relais 1h sur GPU AMD ROCm (`Gemma 2 2B`)

Ce notebook entraîne **`google/gemma-2-2b`** en continu (CPT) sur les micro-shards pré-tokenisés produits par le Notebook 1, en étant conçu dès la base pour survivre aux **sessions éphémères de 1 h** sur GPU AMD (~250 Go VRAM, ex: Instinct MI325X) :

### Architecture validée
- **Modèle de base** : `google/gemma-2-2b` (`bfloat16` natif ROCm, attention `eager` recommandée pour préserver le *logit softcapping* = `50.0`).
- **Paramètres entraînables (~673M)** :
  - **`LoRA r=64, alpha=128` (`use_rslora=True`)** sur `q/k/v/o/gate/up/down_proj` (~83M params, `lr = 1.5e-4`),
  - **`modules_to_save=["embed_tokens", "lm_head"]`** avec **`ensure_weight_tying=True`** (~590M params liés sans duplication, `lr = 3.0e-5`).
- **Mélange intra-batch garanti** : chaque shard téléchargé contient déjà `35 % bci / 35 % dyu / 18 % fr / 12 % en` mélangés aléatoirement (`0 % nch` vérifié par assertion).
- **Relais automatique 1 h** :
  - Reprise exacte au shard et à la ligne (`etat_relais.json` + `etat_optimiseur.pt`),
  - Watchdog à **46 min** : stoppe proprement, évalue la perplexité par langue sur `eval_multilingue.parquet`, puis pousse tout sur Hugging Face avant la coupure à 60 min.

## 1. Environnement AMD ROCm (sans écraser PyTorch HIP)

In [ ]:
import os

# Allocateur ROCm anti-fragmentation et limitation logs MIOpen verbeux
os.environ["PYTORCH_HIP_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MIOPEN_LOG_LEVEL"] = "3"
os.environ["MIOPEN_ENABLE_LOGGING"] = "0"
os.environ["MIOPEN_ENABLE_LOGGING_CMD"] = "0"

In [ ]:
%pip install --quiet --upgrade pip
%pip install --quiet --upgrade --upgrade-strategy only-if-needed "numpy>=2.0.0,<2.8.0" "pyarrow>=17.0.0" "transformers>=4.51.0,<5.0.0" "peft>=0.18.0" "accelerate>=0.34.0" "huggingface_hub>=0.27.0" "safetensors" "tqdm"
%pip install --quiet --force-reinstall --no-deps "transformers>=4.51.0,<5.0.0"

In [ ]:
import numpy as np

if int(np.__version__.split(".")[0]) < 2:
    raise RuntimeError(
        f"[ERREUR] NumPy {np.__version__} détecté en mémoire : SciPy/Transformers exigent numpy>=2.0.0. "
        "Exécutez la cellule %pip install ci-dessus puis redémarrez le kernel Jupyter (Kernel -> Restart)."
    )

import getpass
import inspect
import json
import math
import time
from datetime import datetime, timezone
from pathlib import Path

import peft
import pyarrow.parquet as pq
import torch
import transformers
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download
from huggingface_hub.errors import RepositoryNotFoundError
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

if int(transformers.__version__.split(".")[0]) >= 5:
    raise RuntimeError(
        f"[ERREUR] transformers {transformers.__version__} détecté : exécutez la cellule %pip install (<5.0.0) puis redémarrez le kernel (Kernel -> Restart)."
    )

HEURE_DEBUT_SESSION = time.monotonic()
Path("/root/.local/share/jupyter").mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("[ERREUR] Aucun GPU détecté (CUDA / AMD ROCm HIP requis).")

if "ensure_weight_tying" not in inspect.signature(LoraConfig.__init__).parameters:
    raise ImportError(
        f"[ERREUR] Version de peft ({peft.__version__}) trop ancienne : 'ensure_weight_tying' est requis pour Gemma 2. "
        "Mettez à jour avec peft>=0.18.0 puis redémarrez le kernel Jupyter."
    )

est_rocm = hasattr(torch.version, "hip") and torch.version.hip is not None
bf16_ok = torch.cuda.is_bf16_supported()
if not bf16_ok:
    raise RuntimeError("[ERREUR] Le support bfloat16 natif est requis pour Gemma 2 sur AMD MI3xx.")

nom_gpu = torch.cuda.get_device_name(0)
vram_go = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print("=" * 72)
print(f"[OK] GPU actif         : {nom_gpu} ({vram_go:.1f} Go VRAM)")
print(f"Backend           : {'AMD ROCm HIP ' + str(torch.version.hip) if est_rocm else 'CUDA'}")
print(f"PEFT version      : {peft.__version__} (ensure_weight_tying vérifié)")
print("=" * 72)

## 2. Configuration Centralisée du Relais & Hyperparamètres

In [ ]:
# Authentification Hugging Face
JETON_HF = os.getenv("HF_TOKEN", "").strip()
if not JETON_HF:
    JETON_HF = getpass.getpass("Jeton Hugging Face (lecture + écriture Tree-AI-lab) : ").strip()
if not JETON_HF:
    raise PermissionError("[ERREUR] Jeton HF obligatoire.")
os.environ["HF_TOKEN"] = JETON_HF

# Dépôts Hugging Face
IDENTIFIANT_MODELE_BASE = "google/gemma-2-2b"
DEPOT_SHARDS_HF = "Tree-AI-lab/kouman-gemma2-2b-cpt-shards"
DEPOT_CHECKPOINT_HF = "issoufzousko07/kouman-gemma2-2b-cpt"
DEPOT_REPRISE_INITIALE_HF = "Tree-AI-lab/kouman-gemma2-2b-cpt"
DEPOT_CHECKPOINT_PRIVE = True

# Configuration LoRA + Embeddings liés (~673M params entraînables)
LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.0
MODULES_LORA = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
MODULES_SAUVEGARDE_COMPLETS = ["embed_tokens", "lm_head"]

# Optimisation GPU AMD MI300X 192 Go VRAM (vocab 256k Gemma 2 -> cross_entropy FP32 dimensionnée pour ~80 Go pic)
LONGUEUR_SEQUENCE = 2048
TAILLE_MICRO_LOT = 4               # 4 × 2048 = 8 192 tokens/passe (évite OOM vocab 256k en cross_entropy FP32)
PAS_ACCUMULATION = 32              # Lot effectif inchangé = 128 séquences = 262 144 tokens / step
NB_WORKERS_DATALOADER = min(4, os.cpu_count() or 2)

# Taux d'apprentissage différenciés (embeddings plus doux que LoRA)
TAUX_APPRENTISSAGE_LORA = 1.5e-4
TAUX_APPRENTISSAGE_EMBEDDINGS = 3.0e-5
DECROISSANCE_POIDS_LORA = 0.01
CLIPPING_NORME_GRADIENT = 1.0

# Scheduler WSD (Warmup-Stable-Decay) compatible relais illimités
# - "warmup_stable" : toutes les sessions intermédiaires
# - "decay"         : dernières sessions avant SFT pour converger proprement
PHASE_SCHEDULER = "decay"
PAS_WARMUP = 250
PAS_DECAY_TOTAL = 600              # Utilisé uniquement si PHASE_SCHEDULER == "decay" (5 shards = 640 steps)

# Sauvegarde directe sur HF dès qu'UN SEUL shard est terminé (sans règle d'arrêt à 40 min)
DUREE_MAX_ENTRAINEMENT_MINUTES = None  # None = pas d'arrêt à 40 min, upload automatique à chaque fin de shard
INTERVALLE_LOG_STEPS = 5
INTERVALLE_PUSH_HF_SHARDS = 1          # 1 = upload direct sur HF dès qu'un seul shard finit !
LANGUES_INTERDITES = {"nch"}

DOSSIER_TRAVAIL = Path("travail_cpt_gemma2_relais")
DOSSIER_CHECKPOINT_LOCAL = DOSSIER_TRAVAIL / "checkpoint_courant"
DOSSIER_CACHE_SHARDS = DOSSIER_TRAVAIL / "shards_cache"
DOSSIER_TRAVAIL.mkdir(parents=True, exist_ok=True)
DOSSIER_CHECKPOINT_LOCAL.mkdir(parents=True, exist_ok=True)
DOSSIER_CACHE_SHARDS.mkdir(parents=True, exist_ok=True)

print(f"Modèle base      : {IDENTIFIANT_MODELE_BASE}")
print(f"Shards source    : {DEPOT_SHARDS_HF}")
print(f"Relais HF        : {DEPOT_CHECKPOINT_HF}")
print(f"Lot effectif     : {TAILLE_MICRO_LOT * PAS_ACCUMULATION} séquences ({TAILLE_MICRO_LOT * PAS_ACCUMULATION * LONGUEUR_SEQUENCE:,} tokens/step)")

## 3. Contrats Anti-Fallback, Manifeste Shards & Reprise Relais HF

Nous vérifions rigoureusement :
1. Que le manifeste du Notebook 1 (`manifeste_shards.json`) est accessible et sans Nouchi (`nch`).
2. Côté checkpoint (`DEPOT_CHECKPOINT_HF`) :
   - **Nouveau départ légitime** uniquement si `etat_relais.json` n'existe pas encore dans le dépôt.
   - Si `etat_relais.json` existe, **tous les fichiers critiques** (`adapter_model.safetensors`, `etat_optimiseur.pt`) doivent se télécharger et matcher exactement — sinon erreur explicite bloquante (jamais de réinitialisation silencieuse à step 0).

In [ ]:
class ErreurValidationShards(RuntimeError):
    """Levée si les shards pré-tokenisés sont absents ou incompatibles."""


class ErreurNouchiDetecte(ValueError):
    """Levée si la langue interdite 'nch' est détectée dans un shard."""


class ErreurRepriseCheckpoint(RuntimeError):
    """Levée si la reprise d'un checkpoint relais existant échoue."""


client_hf = HfApi(token=JETON_HF)


def charger_manifeste_shards() -> dict:
    """Télécharge et valide le manifeste généré par le Notebook 1."""
    try:
        chemin_local = hf_hub_download(
            repo_id=DEPOT_SHARDS_HF,
            filename="manifeste_shards.json",
            repo_type="dataset",
            token=JETON_HF,
            force_download=True,
        )
    except Exception as err:
        raise ErreurValidationShards(
            f"[ERREUR] Impossible de lire manifeste_shards.json dans {DEPOT_SHARDS_HF}. "
            "Exécutez d'abord le Notebook 1 !"
        ) from err

    manifeste = json.loads(Path(chemin_local).read_text(encoding="utf-8"))
    if manifeste.get("modele_tokenizer") != IDENTIFIANT_MODELE_BASE:
        raise ErreurValidationShards(
            f"[ERREUR] Tokenizer du manifeste incompatible : {manifeste.get('modele_tokenizer')} != {IDENTIFIANT_MODELE_BASE}"
        )
    if manifeste.get("longueur_sequence") != LONGUEUR_SEQUENCE:
        raise ErreurValidationShards(
            f"[ERREUR] Longueur de séquence incompatible : {manifeste.get('longueur_sequence')} != {LONGUEUR_SEQUENCE}"
        )
    shards = manifeste.get("shards_publies", [])
    if not shards:
        raise ErreurValidationShards("[ERREUR] Aucun shard d'entraînement publié dans manifeste_shards.json !")

    print(
        f"Manifeste shards OK : {len(shards)} shard(s) disponible(s) | "
        f"{manifeste.get('tokens_totaux_publies', 0) / 1e9:.3f}B tokens pré-tokenisés | "
        f"termine={manifeste.get('termine')}"
    )
    return manifeste


def _lister_fichiers_depot(depot_hf: str) -> set[str]:
    """Liste les fichiers d'un dépôt modèle HF ou lève une erreur explicite."""
    try:
        return set(client_hf.list_repo_files(repo_id=depot_hf, repo_type="model", token=JETON_HF))
    except RepositoryNotFoundError as err:
        raise ErreurRepriseCheckpoint(f"[ERREUR] Dépôt checkpoint introuvable : {depot_hf}") from err


def inspecter_etat_relais_hf() -> dict | None:
    """Retourne l'état relais s'il existe sur HF, None s'il s'agit d'un tout premier run."""
    client_hf.create_repo(
        repo_id=DEPOT_CHECKPOINT_HF,
        repo_type="model",
        private=DEPOT_CHECKPOINT_PRIVE,
        exist_ok=True,
    )
    depot_source = DEPOT_CHECKPOINT_HF
    fichiers_repo = _lister_fichiers_depot(depot_source)

    # Si le nouveau dépôt cible n'a pas encore de relais, amorcer depuis le dépôt historique s'il existe
    if (
        "etat_relais.json" not in fichiers_repo
        and DEPOT_REPRISE_INITIALE_HF
        and DEPOT_REPRISE_INITIALE_HF != DEPOT_CHECKPOINT_HF
        and client_hf.repo_exists(repo_id=DEPOT_REPRISE_INITIALE_HF, repo_type="model", token=JETON_HF)
    ):
        fichiers_historique = _lister_fichiers_depot(DEPOT_REPRISE_INITIALE_HF)
        if "etat_relais.json" in fichiers_historique:
            print(
                f"Amorçage depuis {DEPOT_REPRISE_INITIALE_HF} "
                f"(les nouveaux relais seront poussés sur {DEPOT_CHECKPOINT_HF})."
            )
            depot_source = DEPOT_REPRISE_INITIALE_HF
            fichiers_repo = fichiers_historique

    if "etat_relais.json" not in fichiers_repo:
        print("Aucun relais précédent trouvé : initialisation d'un entraînement neuf (step 0).")
        return None

    requis = {
        "checkpoint_courant/adapter_config.json",
        "checkpoint_courant/adapter_model.safetensors",
        "checkpoint_courant/etat_optimiseur.pt",
    }
    manquants = [f for f in sorted(requis) if f not in fichiers_repo]
    if manquants:
        raise ErreurRepriseCheckpoint(
            f"[ERREUR] Relais corrompu sur {depot_source} : etat_relais.json présent mais fichiers manquants : {manquants}"
        )

    chemin_etat = hf_hub_download(
        repo_id=depot_source,
        filename="etat_relais.json",
        repo_type="model",
        token=JETON_HF,
        force_download=True,
    )
    etat = json.loads(Path(chemin_etat).read_text(encoding="utf-8"))

    if "historique_metriques.jsonl" in fichiers_repo:
        chemin_hist_hf = hf_hub_download(
            repo_id=depot_source,
            filename="historique_metriques.jsonl",
            repo_type="model",
            token=JETON_HF,
            force_download=True,
        )
        (DOSSIER_TRAVAIL / "historique_metriques.jsonl").write_bytes(Path(chemin_hist_hf).read_bytes())

    # Télécharge tous les fichiers du dernier checkpoint relais dans DOSSIER_CHECKPOINT_LOCAL
    for chemin_repo in sorted(fichiers_repo):
        if chemin_repo.startswith("checkpoint_courant/"):
            relatif = chemin_repo.removeprefix("checkpoint_courant/")
            cible = DOSSIER_CHECKPOINT_LOCAL / relatif
            cible.parent.mkdir(parents=True, exist_ok=True)
            telecharge = hf_hub_download(
                repo_id=depot_source,
                filename=chemin_repo,
                repo_type="model",
                token=JETON_HF,
                force_download=True,
            )
            cible.write_bytes(Path(telecharge).read_bytes())

    print(
        f"Relais repris depuis {depot_source} : step_global={etat['step_global']} | "
        f"shard={etat['indice_shard']} | ligne={etat['ligne_dans_shard']} | "
        f"tokens_vus={etat['tokens_entraines'] / 1e9:.3f}B"
    )
    return etat


MANIFESTE_SHARDS = charger_manifeste_shards()
etat_relais_precedent = inspecter_etat_relais_hf()

## 4. Chargement Gemma 2 2B + PEFT LoRA & Poids Liés (`ensure_weight_tying=True`)

Point critique sur **Gemma 2 + ROCm** :
- `attn_implementation="eager"` est recommandé par `transformers` pour appliquer exactement le *logit softcapping* (`50.0`) natif de Gemma 2 pendant l'entraînement.
- Assertion stricte sur le nombre de paramètres entraînables (`600M <= N <= 750M`) pour vérifier que `embed_tokens` et `lm_head` restent bien liés (sinon le dédoublement monterait à ~1,26B).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(IDENTIFIANT_MODELE_BASE, token=JETON_HF)

print(f"Chargement du modèle base {IDENTIFIANT_MODELE_BASE} en bfloat16 (attn=eager)...")
modele_base = AutoModelForCausalLM.from_pretrained(
    IDENTIFIANT_MODELE_BASE,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    token=JETON_HF,
)
modele_base.config.use_cache = False
if vram_go < 100.0:
    print(f"GPU {vram_go:.0f} Go détecté (< 100 Go) : activation du Gradient Checkpointing pour éviter tout OOM sur vocab 256k.")
    modele_base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    modele_base.enable_input_require_grads()

if etat_relais_precedent is None:
    config_lora = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        use_rslora=True,
        bias="none",
        target_modules=MODULES_LORA,
        modules_to_save=MODULES_SAUVEGARDE_COMPLETS,
        ensure_weight_tying=True,
        task_type=TaskType.CAUSAL_LM,
    )
    modele = get_peft_model(modele_base, config_lora)
else:
    modele = PeftModel.from_pretrained(
        modele_base,
        str(DOSSIER_CHECKPOINT_LOCAL),
        is_trainable=True,
    )

modele = modele.to(device="cuda", dtype=torch.bfloat16)

params_totaux = sum(p.numel() for p in modele.parameters())
params_entrainables = sum(p.numel() for p in modele.parameters() if p.requires_grad)
ratio_pct = 100.0 * params_entrainables / params_totaux

print(f"Paramètres totaux       : {params_totaux:,}")
print(f"Paramètres entraînables : {params_entrainables:,} ({ratio_pct:.2f} %)")

if not (600_000_000 <= params_entrainables <= 750_000_000):
    raise AssertionError(
        f"[ERREUR] Nombre de paramètres entraînables inattendu ({params_entrainables:,}). "
        "Vérifiez que ensure_weight_tying=True lie bien embed_tokens et lm_head (~673M attendu)."
    )

if etat_relais_precedent is not None:
    attendu = etat_relais_precedent.get("params_entrainables")
    if attendu is not None and int(attendu) != int(params_entrainables):
        raise ErreurRepriseCheckpoint(
            f"[ERREUR] Nombre de paramètres différent du checkpoint précédent : {params_entrainables} != {attendu}"
        )

## 5. Optimiseur Différencié & Scheduler WSD (`Warmup-Stable-Decay`)

Les matrices `LoRA` reçoivent `1.5e-4`, tandis que les embeddings liés `embed_tokens`/`lm_head` reçoivent `3.0e-5` (5× plus doux) pour assimiler le vocabulaire Baoulé/Dioula sans désaligner les connaissances Français/Anglais de Gemma 2.

In [ ]:
params_embeddings = []
params_lora = []
noms_parametres_vus = set()

for nom, param in modele.named_parameters():
    if not param.requires_grad:
        continue
    # Évite de compter deux fois le même tenseur si partagé par weight tying
    if id(param) in noms_parametres_vus:
        continue
    noms_parametres_vus.add(id(param))
    if "embed_tokens" in nom or "lm_head" in nom:
        params_embeddings.append(param)
    else:
        params_lora.append(param)

if not params_embeddings or not params_lora:
    raise RuntimeError(
        f"[ERREUR] Groupes de paramètres incomplets : embeddings={len(params_embeddings)}, lora={len(params_lora)}"
    )

groupes_optimiseur = [
    {
        "params": params_lora,
        "lr": TAUX_APPRENTISSAGE_LORA,
        "weight_decay": DECROISSANCE_POIDS_LORA,
        "lr_cible_base": TAUX_APPRENTISSAGE_LORA,
    },
    {
        "params": params_embeddings,
        "lr": TAUX_APPRENTISSAGE_EMBEDDINGS,
        "weight_decay": 0.0,
        "lr_cible_base": TAUX_APPRENTISSAGE_EMBEDDINGS,
    },
]

optimiseur = torch.optim.AdamW(groupes_optimiseur, betas=(0.9, 0.95), eps=1e-8)

if etat_relais_precedent is not None:
    chemin_opt = DOSSIER_CHECKPOINT_LOCAL / "etat_optimiseur.pt"
    etat_opt = torch.load(chemin_opt, map_location="cpu", weights_only=True)
    optimiseur.load_state_dict(etat_opt["optimizer_state_dict"])
    step_global = int(etat_relais_precedent["step_global"])
    indice_shard_courant = int(etat_relais_precedent["indice_shard"])
    ligne_dans_shard_courant = int(etat_relais_precedent["ligne_dans_shard"])
    tokens_entraines = int(etat_relais_precedent["tokens_entraines"])
    step_debut_decay = etat_relais_precedent.get("step_debut_decay")
    print(f"États AdamW restaurés au step {step_global:,} !")
else:
    step_global = 0
    indice_shard_courant = 0
    ligne_dans_shard_courant = 0
    tokens_entraines = 0
    step_debut_decay = None

if PHASE_SCHEDULER == "decay" and step_debut_decay is None:
    step_debut_decay = step_global
    print(f"Activation de la phase DECAY WSD à partir du step {step_debut_decay}.")


def appliquer_facteur_wsd(step_actuel: int) -> float:
    """Calcule le multiplicateur LR Warmup-Stable-Decay et l'applique à l'optimiseur."""
    if step_actuel < PAS_WARMUP:
        facteur = float(step_actuel + 1) / float(max(1, PAS_WARMUP))
    elif PHASE_SCHEDULER == "warmup_stable":
        facteur = 1.0
    elif PHASE_SCHEDULER == "decay":
        debut = step_debut_decay if step_debut_decay is not None else step_actuel
        avancement = min(1.0, max(0.0, (step_actuel - debut) / float(max(1, PAS_DECAY_TOTAL))))
        # Descente cosine douce jusqu'à 10 % du LR initial
        facteur = 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * avancement))
    else:
        raise ValueError(f"[ERREUR] PHASE_SCHEDULER inconnue : {PHASE_SCHEDULER}")

    for groupe in optimiseur.param_groups:
        groupe["lr"] = groupe["lr_cible_base"] * facteur
    return facteur

## 6. Chargement des Shards & Évaluation Multilingue Figée

In [ ]:
class JeuShardPret(Dataset):
    """Dataset PyTorch en mémoire zéro-copie pour un micro-shard Parquet (int32 -> int64)."""

    def __init__(self, chemin_parquet: Path, ligne_depart: int = 0):
        table = pq.read_table(chemin_parquet, columns=["input_ids", "lang"])
        langues = table["lang"].to_pylist()
        if "nch" in set(langues):
            raise ErreurNouchiDetecte(f"[ERREUR] Shard contaminé par du Nouchi ('nch') : {chemin_parquet}")

        # Conversion rapide en tenseur PyTorch int64 [N, LONGUEUR_SEQUENCE]
        tenseur_complet = torch.tensor(table["input_ids"].to_pylist(), dtype=torch.int64)
        if tenseur_complet.ndim != 2 or tenseur_complet.shape[1] != LONGUEUR_SEQUENCE:
            raise ErreurValidationShards(
                f"[ERREUR] Dimensions shard invalides : {tuple(tenseur_complet.shape)} != (*, {LONGUEUR_SEQUENCE})"
            )
        if ligne_depart > len(tenseur_complet):
            raise ErreurRepriseCheckpoint(
                f"[ERREUR] ligne_depart ({ligne_depart}) > taille du shard ({len(tenseur_complet)})"
            )
        self.sequences = tenseur_complet[ligne_depart:]

    def __len__(self) -> int:
        return self.sequences.shape[0]

    def __getitem__(self, idx: int) -> torch.Tensor:
        return self.sequences[idx]


def obtenir_shard_local(chemin_repo_shard: str) -> Path:
    """Télécharge le shard demandé depuis Hugging Face vers le cache local."""
    telecharge = hf_hub_download(
        repo_id=DEPOT_SHARDS_HF,
        filename=chemin_repo_shard,
        repo_type="dataset",
        token=JETON_HF,
        local_dir=str(DOSSIER_CACHE_SHARDS),
    )
    return Path(telecharge)


def preparer_batches_eval() -> dict[str, torch.Tensor]:
    """Charge le split d'évaluation figé regroupé par langue (bci, dyu, fr, en)."""
    chemin_eval = obtenir_shard_local("eval/eval_multilingue.parquet")
    table_eval = pq.read_table(chemin_eval, columns=["input_ids", "lang"])
    toutes_seqs = table_eval["input_ids"].to_pylist()
    toutes_langs = table_eval["lang"].to_pylist()
    if "nch" in set(toutes_langs):
        raise ErreurNouchiDetecte("[ERREUR] Nouchi détecté dans eval_multilingue.parquet !")

    par_langue: dict[str, list[list[int]]] = {}
    for seq, lg in zip(toutes_seqs, toutes_langs):
        par_langue.setdefault(lg, []).append(seq)

    return {lg: torch.tensor(seqs, dtype=torch.int64) for lg, seqs in par_langue.items()}


EVAL_PAR_LANGUE = preparer_batches_eval()
print("Jeu d'évaluation figé chargé :", {lg: t.shape[0] for lg, t in EVAL_PAR_LANGUE.items()})


@torch.no_grad()
def evaluer_perplexite_par_langue(max_seqs_par_langue: int = 128) -> dict[str, dict[str, float]]:
    """Mesure la loss et la perplexité exacte séparément sur bci, dyu, fr, en."""
    modele.eval()
    resultats = {}
    for lg, tenseur in EVAL_PAR_LANGUE.items():
        sous_ensemble = tenseur[:max_seqs_par_langue]
        pertes = []
        for debut in range(0, len(sous_ensemble), TAILLE_MICRO_LOT):
            lot = sous_ensemble[debut:debut + TAILLE_MICRO_LOT].to("cuda", non_blocking=True)
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                sortie = modele(input_ids=lot, labels=lot)
            pertes.append(float(sortie.loss.detach().cpu()))
            del lot, sortie
        loss_moy = sum(pertes) / len(pertes)
        resultats[lg] = {
            "loss": round(loss_moy, 4),
            "ppl": round(math.exp(min(loss_moy, 20.0)), 2),
        }
    torch.cuda.empty_cache()
    modele.train()
    return resultats

## 7. Fonction de Sauvegarde Atomique Relais & Boucle d'Entraînement 1h

In [ ]:
def temps_ecoule_minutes() -> float:
    return (time.monotonic() - HEURE_DEBUT_SESSION) / 60.0


def sauvegarder_et_pousser_relais(
    indice_shard: int,
    ligne_dans_shard: int,
    motif_arret: str,
    metriques_eval: dict,
) -> None:
    """Sauvegarde adaptateurs + embeddings + AdamW + curseur et pousse vers HF."""
    print(f"Sauvegarde relais ({motif_arret}) à step={step_global:,} | shard={indice_shard} | ligne={ligne_dans_shard}...")

    # 1. Adaptateurs LoRA + embeddings liés
    modele.save_pretrained(str(DOSSIER_CHECKPOINT_LOCAL))
    tokenizer.save_pretrained(str(DOSSIER_CHECKPOINT_LOCAL))

    # 2. États complets de l'optimiseur AdamW
    torch.save(
        {"optimizer_state_dict": optimiseur.state_dict()},
        DOSSIER_CHECKPOINT_LOCAL / "etat_optimiseur.pt",
    )

    # 3. État du curseur relais JSON
    etat_relais = {
        "modele_base": IDENTIFIANT_MODELE_BASE,
        "step_global": step_global,
        "indice_shard": indice_shard,
        "ligne_dans_shard": ligne_dans_shard,
        "tokens_entraines": tokens_entraines,
        "params_entrainables": params_entrainables,
        "phase_scheduler": PHASE_SCHEDULER,
        "step_debut_decay": step_debut_decay,
        "motif_dernier_arret": motif_arret,
        "dernieres_metriques_eval": metriques_eval,
        "horodatage_utc": datetime.now(timezone.utc).isoformat(),
    }
    chemin_etat_local = DOSSIER_TRAVAIL / "etat_relais.json"
    chemin_etat_local.write_text(json.dumps(etat_relais, indent=2, ensure_ascii=False), encoding="utf-8")

    # 4. Historique cumulatif des métriques
    chemin_hist = DOSSIER_TRAVAIL / "historique_metriques.jsonl"
    with open(chemin_hist, "a", encoding="utf-8") as f_hist:
        f_hist.write(
            json.dumps(
                {
                    "step_global": step_global,
                    "tokens_entraines": tokens_entraines,
                    "motif": motif_arret,
                    "eval": metriques_eval,
                    "utc": etat_relais["horodatage_utc"],
                },
                ensure_ascii=False,
            )
            + "\n"
        )

    # 5. Commit atomique unique sur Hugging Face (checkpoint + état + historique)
    operations = []
    for fichier_ckpt in sorted(DOSSIER_CHECKPOINT_LOCAL.rglob("*")):
        if fichier_ckpt.is_file():
            relatif = fichier_ckpt.relative_to(DOSSIER_CHECKPOINT_LOCAL).as_posix()
            operations.append(
                CommitOperationAdd(
                    path_or_fileobj=str(fichier_ckpt),
                    path_in_repo=f"checkpoint_courant/{relatif}",
                )
            )
    operations.append(CommitOperationAdd(path_or_fileobj=str(chemin_etat_local), path_in_repo="etat_relais.json"))
    operations.append(CommitOperationAdd(path_or_fileobj=str(chemin_hist), path_in_repo="historique_metriques.jsonl"))

    # 5. Purge préventive si plus d'un commit existe déjà, commit atomique, puis squash LFS
    commits_existants = client_hf.list_repo_commits(repo_id=DEPOT_CHECKPOINT_HF, repo_type="model")
    if len(commits_existants) > 1:
        client_hf.super_squash_history(
            repo_id=DEPOT_CHECKPOINT_HF,
            repo_type="model",
            branch="main",
            commit_message="squash historique avant nouveau relais",
        )

    message_commit = (
        f"checkpoint atomique: step {step_global} "
        f"({tokens_entraines / 1e9:.3f}B tokens) [{motif_arret}]"
    )
    client_hf.create_commit(
        repo_id=DEPOT_CHECKPOINT_HF,
        repo_type="model",
        operations=operations,
        commit_message=message_commit,
    )
    client_hf.super_squash_history(
        repo_id=DEPOT_CHECKPOINT_HF,
        repo_type="model",
        branch="main",
        commit_message=message_commit,
    )
    print(f"Historique LFS compressé sur HF (1 commit conservé pour step={step_global}).")
    print(f"Relais synchronisé atomiquement sur HF ({temps_ecoule_minutes():.1f} min écoulées).")


# Baseline initiale uniquement au step 0
if step_global == 0:
    print("Mesure baseline avant le premier step d'entraînement...")
    eval_initiale = evaluer_perplexite_par_langue()
    print("Perplexité initiale Gemma 2 2B :", eval_initiale)

shards_disponibles = MANIFESTE_SHARDS["shards_publies"]
arret_watchdog = False
besoin_push_relais = False
modele.train()
optimiseur.zero_grad(set_to_none=True)
compteur_accum = 0
perte_accum = 0.0

while indice_shard_courant < len(shards_disponibles):
    if DUREE_MAX_ENTRAINEMENT_MINUTES is not None and temps_ecoule_minutes() >= DUREE_MAX_ENTRAINEMENT_MINUTES:
        arret_watchdog = True
        break

    info_shard = shards_disponibles[indice_shard_courant]
    chemin_parquet_shard = obtenir_shard_local(info_shard["fichier"])
    nb_total_lignes_shard = int(info_shard["nb_sequences"])

    jeu_shard = JeuShardPret(chemin_parquet_shard, ligne_depart=ligne_dans_shard_courant)
    chargeur = DataLoader(
        jeu_shard,
        batch_size=TAILLE_MICRO_LOT,
        shuffle=False,  # Déjà mélangé aléatoirement à l'intérieur du shard lors du Notebook 1
        num_workers=NB_WORKERS_DATALOADER,
        pin_memory=True,
        drop_last=False,
    )

    print(
        f"Shard {indice_shard_courant + 1}/{len(shards_disponibles)} : {info_shard['fichier']} "
        f"(à partir de la ligne {ligne_dans_shard_courant}/{nb_total_lignes_shard})"
    )
    barre = tqdm(chargeur, desc=f"Shard {indice_shard_courant}", leave=False)

    for lot_ids in barre:
        nb_seqs_lot = int(lot_ids.shape[0])
        lot_cuda = lot_ids.to("cuda", non_blocking=True)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            sortie = modele(input_ids=lot_cuda, labels=lot_cuda)
            perte_brute = sortie.loss
            perte_normalisee = perte_brute / PAS_ACCUMULATION

        perte_normalisee.backward()
        perte_accum += float(perte_brute.detach().cpu())
        compteur_accum += 1
        ligne_dans_shard_courant += nb_seqs_lot
        tokens_entraines += nb_seqs_lot * LONGUEUR_SEQUENCE

        if compteur_accum >= PAS_ACCUMULATION:
            facteur_lr = appliquer_facteur_wsd(step_global)
            torch.nn.utils.clip_grad_norm_(
                [p for g in optimiseur.param_groups for p in g["params"]],
                CLIPPING_NORME_GRADIENT,
            )
            optimiseur.step()
            optimiseur.zero_grad(set_to_none=True)

            loss_step = perte_accum / compteur_accum
            perte_accum = 0.0
            compteur_accum = 0
            step_global += 1

            if step_global % INTERVALLE_LOG_STEPS == 0:
                lr_actuel = optimiseur.param_groups[0]["lr"]
                barre.set_postfix(
                    step=step_global,
                    loss=f"{loss_step:.3f}",
                    lr=f"{lr_actuel:.2e}",
                    min=f"{temps_ecoule_minutes():.1f}",
                )

            if DUREE_MAX_ENTRAINEMENT_MINUTES is not None and temps_ecoule_minutes() >= DUREE_MAX_ENTRAINEMENT_MINUTES:
                print(
                    f"Watchdog atteint à {temps_ecoule_minutes():.1f} min "
                    "(après un optimizer.step propre). Sauvegarde relais..."
                )
                arret_watchdog = True
                break

    if arret_watchdog:
        optimiseur.zero_grad(set_to_none=True)
        compteur_accum = 0
        perte_accum = 0.0
        eval_courante = evaluer_perplexite_par_langue()
        print("Évaluation relais :", eval_courante)
        sauvegarder_et_pousser_relais(
            indice_shard=indice_shard_courant,
            ligne_dans_shard=ligne_dans_shard_courant,
            motif_arret="watchdog_1h",
            metriques_eval=eval_courante,
        )
        besoin_push_relais = False
        break

    # Shard terminé proprement : purge des éventuels restes d'accumulation (< PAS_ACCUMULATION)
    optimiseur.zero_grad(set_to_none=True)
    compteur_accum = 0
    perte_accum = 0.0

    indice_shard_courant += 1
    ligne_dans_shard_courant = 0
    chemin_parquet_shard.unlink(missing_ok=True)

    # Upload DIRECT et systématique sur Hugging Face dès que ce shard est terminé
    eval_shard = evaluer_perplexite_par_langue()
    print(f"[OK] Shard {indice_shard_courant - 1} terminé | Éval={eval_shard} -> Upload direct sur HF...")
    est_dernier_shard = indice_shard_courant >= len(shards_disponibles)
    decay_termine = (
        PHASE_SCHEDULER == "decay"
        and step_debut_decay is not None
        and (step_global - step_debut_decay) >= PAS_DECAY_TOTAL
    )
    if decay_termine:
        motif_shard = "fin_decay_cpt"
    elif est_dernier_shard:
        motif_shard = "fin_shards"
    else:
        motif_shard = f"shard_{indice_shard_courant - 1}_termine"

    sauvegarder_et_pousser_relais(
        indice_shard=indice_shard_courant,
        ligne_dans_shard=0,
        motif_arret=motif_shard,
        metriques_eval=eval_shard,
    )
    besoin_push_relais = False

    if decay_termine:
        print(
            f"Phase DECAY WSD terminée ({step_global - step_debut_decay} steps >= {PAS_DECAY_TOTAL}) ! "
            "Le CPT est prêt pour le test final et la fusion (sections suivantes)."
        )
        break

if besoin_push_relais:
    eval_finale_session = evaluer_perplexite_par_langue()
    sauvegarder_et_pousser_relais(
        indice_shard=indice_shard_courant,
        ligne_dans_shard=ligne_dans_shard_courant,
        motif_arret="watchdog_fin_boucle",
        metriques_eval=eval_finale_session,
    )

if not arret_watchdog and indice_shard_courant >= len(shards_disponibles):
    print("Tous les shards disponibles dans manifeste_shards.json ont été entraînés !")

## 8. (Optionnel — Fin du CPT uniquement) Fusion LoRA + Embeddings & Export `bfloat16`

À exécuter **uniquement** lorsque toutes les sessions (y compris la phase `PHASE_SCHEDULER = "decay"`) sont terminées pour obtenir le modèle de base Kouman IA fusionné prêt pour le SFT :

In [ ]:
FUSIONNER_ET_PUBLIER_MODELE_FINAL = False  # Mettre à True uniquement tout à la fin du CPT
DEPOT_MODELE_FUSIONNE = "issoufzousko07/kouman-gemma2-2b-cpt-merged"

if FUSIONNER_ET_PUBLIER_MODELE_FINAL:
    print("Fusion propre LoRA + embeddings liés (ensure_weight_tying)...")
    modele_fusionne = modele.merge_and_unload()
    dossier_fusion = DOSSIER_TRAVAIL / "modele_fusionne_bf16"
    dossier_fusion.mkdir(parents=True, exist_ok=True)
    modele_fusionne.save_pretrained(str(dossier_fusion), safe_serialization=True)
    tokenizer.save_pretrained(str(dossier_fusion))

    client_hf.create_repo(repo_id=DEPOT_MODELE_FUSIONNE, repo_type="model", private=True, exist_ok=True)
    client_hf.upload_folder(
        folder_path=str(dossier_fusion),
        repo_id=DEPOT_MODELE_FUSIONNE,
        repo_type="model",
        commit_message="feat: modèle Gemma 2 2B CPT fusionné (35% bci, 35% dyu, 18% fr, 12% en, 0% nch)",
    )
    print(f"Modèle CPT complet prêt pour le SFT sur : https://huggingface.co/{DEPOT_MODELE_FUSIONNE}")
else:
    print("[INFO] Fusion désactivée (FUSIONNER_ET_PUBLIER_MODELE_FINAL=False). Continuez les sessions relais si nécessaire.")